In [ ]:
-- ============================================================
-- stg_events.sql
-- Staging layer: cleans and standardises raw event data
-- ============================================================

WITH raw_events AS (
    SELECT
        ORGANIZATION_ID,
        ACTIVITY_NAME,
        TIMESTAMP,
        CONVERTED,
        CONVERTED_AT,
        TRIAL_START,
        TRIAL_END
    FROM events
),

cleaned AS (
    SELECT
        ORGANIZATION_ID,
        ACTIVITY_NAME,

        -- Parse timestamps safely
        TRY_CAST(TIMESTAMP AS TIMESTAMP)    AS event_timestamp,
        TRY_CAST(TRIAL_START AS TIMESTAMP)  AS trial_start,
        TRY_CAST(TRIAL_END AS TIMESTAMP)    AS trial_end,
        TRY_CAST(CONVERTED_AT AS TIMESTAMP) AS converted_at,

        CONVERTED AS is_converted,

        -- Derived fields
        DATEDIFF(
            'day',
            TRY_CAST(TRIAL_START AS TIMESTAMP),
            TRY_CAST(TIMESTAMP AS TIMESTAMP)
        ) AS days_into_trial

    FROM raw_events

    -- Remove duplicates
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY ORGANIZATION_ID, ACTIVITY_NAME, TIMESTAMP
        ORDER BY TIMESTAMP
    ) = 1

    -- Only keep events within trial window
    WHERE TRY_CAST(TIMESTAMP AS TIMESTAMP) IS NOT NULL
      AND TRY_CAST(TIMESTAMP AS TIMESTAMP) >= TRY_CAST(TRIAL_START AS TIMESTAMP)
      AND TRY_CAST(TIMESTAMP AS TIMESTAMP) <= TRY_CAST(TRIAL_END AS TIMESTAMP)
)

SELECT * FROM cleaned